# 01 — Product Backorder Prediction

Portfolio edition of the original course analysis. Outputs are historical and were not rerun. See `reports/FINDINGS.md` for limitations. Code edits address imports, stale variable names and local data paths only.

Part 3 retrains on the saved LOF-filtered development subset, not the full smart sample.

---

# Part 1: Data Preprocessing

In this part, we preprocess the given training set. 


In [79]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['NUMEXPR_NUM_THREADS'] = '2'

%matplotlib inline
import matplotlib.pyplot as plt

import os, sys
import itertools
import numpy as np
import pandas as pd
import joblib

## Load dataset

**Description**
~~~
sku - Random ID for the product
national_inv - Current inventory level for the part
lead_time - Transit time for product (if available)
in_transit_qty - Amount of product in transit from source
forecast_3_month - Forecast sales for the next 3 months
forecast_6_month - Forecast sales for the next 6 months
forecast_9_month - Forecast sales for the next 9 months
sales_1_month - Sales quantity for the prior 1 month time period
sales_3_month - Sales quantity for the prior 3 month time period
sales_6_month - Sales quantity for the prior 6 month time period
sales_9_month - Sales quantity for the prior 9 month time period
min_bank - Minimum recommend amount to stock
potential_issue - Source issue for part identified
pieces_past_due - Parts overdue from source
perf_6_month_avg - Source performance for prior 6 month period
perf_12_month_avg - Source performance for prior 12 month period
local_bo_qty - Amount of stock orders overdue
deck_risk - Part risk flag
oe_constraint - Part risk flag
ppap_risk - Part risk flag
stop_auto_buy - Part risk flag
rev_stop - Part risk flag
went_on_backorder - Product actually went on backorder. 
~~~

**NOTE**: This is a real-world dataset without any preprocessing. There will also be warnings due to fact that the 1st column is mixing integer and string values. The last column, `went_on_backorder`, is what we are trying to predict.


In [80]:
# location of the training set; for Parts 1 and 2, 
# you only have access to this training data set for Parts 1 and 2 
DATASET = os.path.join(os.environ.get('BACKORDER_DATA_DIR', 'data'), 'Kaggle_Training_Dataset_v2.csv')
assert os.path.exists(DATASET)

# Load and shuffle
dataset = pd.read_csv(DATASET,dtype={'sku': str}).sample(frac=1).reset_index(drop=True)
dataset.head().transpose()

,0,1,2,3,4
sku,1547008,1176684,1776790,3181173,3252118
national_inv,3.0,52.0,247.0,6.0,1.0
lead_time,8.0,9.0,12.0,8.0,8.0
in_transit_qty,2.0,0.0,0.0,0.0,0.0
forecast_3_month,2.0,0.0,0.0,0.0,1.0
forecast_6_month,10.0,0.0,0.0,0.0,1.0
forecast_9_month,14.0,7.0,0.0,0.0,2.0
sales_1_month,0.0,7.0,0.0,0.0,0.0
sales_3_month,3.0,22.0,3.0,0.0,1.0
sales_6_month,11.0,39.0,3.0,4.0,1.0


In [81]:
dataset.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
national_inv,1687860.0,496.111782,29615.233831,-27256.0,4.00,15.00,80.00,12334404.0
lead_time,1586967.0,7.872267,7.056024,0.0,4.00,8.00,9.00,52.0
in_transit_qty,1687860.0,44.052022,1342.741731,0.0,0.00,0.00,0.00,489408.0
forecast_3_month,1687860.0,178.119284,5026.553102,0.0,0.00,0.00,4.00,1427612.0
forecast_6_month,1687860.0,344.986664,9795.151862,0.0,0.00,0.00,12.00,2461360.0
forecast_9_month,1687860.0,506.364431,14378.923562,0.0,0.00,0.00,20.00,3777304.0
sales_1_month,1687860.0,55.926069,1928.195879,0.0,0.00,0.00,4.00,741774.0
sales_3_month,1687860.0,175.025930,5192.377625,0.0,0.00,1.00,15.00,1105478.0
sales_6_month,1687860.0,341.728839,9613.167104,0.0,0.00,2.00,31.00,2146625.0
sales_9_month,1687860.0,525.269701,14838.613523,0.0,0.00,4.00,47.00,3205172.0


---

## Processing

In this section, the goal is to figure out:

* which columns we can use directly,  
* which columns are usable after some processing,  
* and which columns are not processable or obviously irrelevant (like product id) that we will discard.

Then process and prepare this dataset for creating a predictive model.

**You can choose to employ any kind of approach for Exploratory Data Analysis to understand the data better. It is up to you to make the decisions regarding the cleaning/reencoding/dropping/imputing/binning etc.**  The following cells only serve as guidance. 

In [82]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1687861 entries, 0 to 1687860
Data columns (total 23 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   sku                1687861 non-null  object 
 1   national_inv       1687860 non-null  float64
 2   lead_time          1586967 non-null  float64
 3   in_transit_qty     1687860 non-null  float64
 4   forecast_3_month   1687860 non-null  float64
 5   forecast_6_month   1687860 non-null  float64
 6   forecast_9_month   1687860 non-null  float64
 7   sales_1_month      1687860 non-null  float64
 8   sales_3_month      1687860 non-null  float64
 9   sales_6_month      1687860 non-null  float64
 10  sales_9_month      1687860 non-null  float64
 11  min_bank           1687860 non-null  float64
 12  potential_issue    1687860 non-null  object 
 13  pieces_past_due    1687860 non-null  float64
 14  perf_6_month_avg   1687860 non-null  float64
 15  perf_12_month_avg  1687860 non-n

### Take samples and examine the dataset

In [83]:
dataset.iloc[:3,:6]

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month
0,1547008,3.0,8.0,2.0,2.0,10.0
1,1176684,52.0,9.0,0.0,0.0,0.0
2,1776790,247.0,12.0,0.0,0.0,0.0


In [84]:
dataset.iloc[:3,6:12]

,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,sales_9_month,min_bank
0,14.0,0.0,3.0,11.0,13.0,3.0
1,7.0,7.0,22.0,39.0,61.0,28.0
2,0.0,0.0,3.0,3.0,3.0,1.0


In [85]:
dataset.iloc[:3,12:18]

,potential_issue,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,deck_risk
0,No,0.0,0.74,0.84,0.0,No
1,No,0.0,0.99,0.98,0.0,No
2,No,0.0,0.58,0.58,0.0,Yes


In [86]:
dataset.iloc[:3,18:24]

,oe_constraint,ppap_risk,stop_auto_buy,rev_stop,went_on_backorder
0,No,No,Yes,No,No
1,No,No,Yes,No,No
2,No,No,Yes,No,No


### Drop columns that are obviously irrelevant or not processable

In [87]:
# ----------------------------------

#The columns I decided to remove is sku: seems like its just some ID number

dataset = dataset.drop(columns=['sku'])


### Find unique values of string columns

Now, try to make sure that the Yes/No columns really only contain Yes or No.  
If that's true, proceed to convert them into binaries (0s and 1s).

**Tip**: use [unique()](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.Series.unique.html) function of pandas Series.

Example

~~~python
print('went_on_backorder', dataset['went_on_backorder'].unique())
~~~

In [88]:
# All the column names of these yes/no columns
yes_no_columns = list(filter(lambda i: dataset[i].dtype!=np.float64, dataset.columns))
print(yes_no_columns)

# ----------------------------------
for column in yes_no_columns:
    print(column, dataset[column].unique())


['potential_issue', 'deck_risk', 'oe_constraint', 'ppap_risk', 'stop_auto_buy', 'rev_stop', 'went_on_backorder']
potential_issue ['No' 'Yes' nan]
deck_risk ['No' 'Yes' nan]
oe_constraint ['No' 'Yes' nan]
ppap_risk ['No' 'Yes' nan]
stop_auto_buy ['No' 'Yes' nan]
rev_stop ['No' 'Yes' nan]
went_on_backorder ['No' 'Yes' nan]


You may possibly also see **nan** representing missing values in the dataset.

We can fill them using the most popular values, that is the Mode in Statistics.

In [89]:
for column_name in yes_no_columns:
    mode = dataset[column_name].apply(str).mode()[0]
    print('Filling missing values of {} with {}'.format(column_name, mode))
    dataset[column_name] = dataset[column_name].fillna(mode)
dataset = dataset.fillna(dataset.median(numeric_only=True))

Filling missing values of potential_issue with No
Filling missing values of deck_risk with No
Filling missing values of oe_constraint with No
Filling missing values of ppap_risk with No
Filling missing values of stop_auto_buy with Yes
Filling missing values of rev_stop with No
Filling missing values of went_on_backorder with No


### Convert yes/no columns into binary (0s and 1s)

In [90]:
# ----------------------------------

for column in yes_no_columns:
    dataset[column] = dataset[column].map({'Yes': 1, 'No': 0})

    


In [91]:
#verify the binary replacement
for column in yes_no_columns:
    print(column, dataset[column].unique())

potential_issue [0 1]
deck_risk [0 1]
oe_constraint [0 1]
ppap_risk [0 1]
stop_auto_buy [0 1]
rev_stop [0 1]
went_on_backorder [0 1]


Now all columns should be either int64 or float64.

In [92]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1687861 entries, 0 to 1687860
Data columns (total 22 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   national_inv       1687861 non-null  float64
 1   lead_time          1687861 non-null  float64
 2   in_transit_qty     1687861 non-null  float64
 3   forecast_3_month   1687861 non-null  float64
 4   forecast_6_month   1687861 non-null  float64
 5   forecast_9_month   1687861 non-null  float64
 6   sales_1_month      1687861 non-null  float64
 7   sales_3_month      1687861 non-null  float64
 8   sales_6_month      1687861 non-null  float64
 9   sales_9_month      1687861 non-null  float64
 10  min_bank           1687861 non-null  float64
 11  potential_issue    1687861 non-null  int64  
 12  pieces_past_due    1687861 non-null  float64
 13  perf_6_month_avg   1687861 non-null  float64
 14  perf_12_month_avg  1687861 non-null  float64
 15  local_bo_qty       1687861 non-n

### Smartly sample the data into a more manageable size for cross-validation 


This is a good point to re-balance dataset before actually moving on. For sampling, we can either take advantage of pandas/numpy `sample` method or use `imblearn` [package](https://imbalanced-learn.org/stable/user_guide.html#user-guide). 


Create a "smart", balanced sample: 

 * it should be balanced: it is up to you if you do a 50/50 balance or anything else. 
 * it should be manageable-sized: you will run a lot of cross validations in Part 2 that are very time consuming. 
 * it should still reflect the characteristics of the original data.
 
Use any approach you deem necessary to create the balanced sample. It will serve as your data set in the development.  



In [93]:
num_backorder = np.sum(dataset['went_on_backorder']==1)
print('backorder ratio:', num_backorder, '/', len(dataset), '=', num_backorder / len(dataset))

backorder ratio: 11293 / 1687861 = 0.006690716830355106


Create a smart sample of the data and save it.  You can either store the data to csv files or simply use `joblib` to dump the variables and load them in Part 2. 

**Example code for using joblib:**

Say we need to store three objects (sampled_X, sampled_y, model) to a file. 

```python
import joblib

# for dumping 
joblib.dump([sampled_X, sampled_y, model], 'data/sample-data-v1.pkl')

# for loading
sampled_X, sampled_y, model = joblib.load('data/sample-data-v1.pkl')

```


In [94]:
# ----------------------------------
#The dataset is highly imbalanced with 0.67% of products foing on backorder.I kept all backorder cases and smapled non-backorder cases at a 3:1 ratio.

backorder = dataset[dataset['went_on_backorder']==1]
no_backorder = dataset[dataset['went_on_backorder']==0]
sample = pd.concat([backorder, no_backorder.sample(n=3*len(backorder), random_state=42)
                   ]).sample(frac=1, random_state=42).reset_index(drop=True)
cnt = sample['went_on_backorder'].value_counts()
print(cnt/len(sample))




went_on_backorder
0    0.75
1    0.25
Name: count, dtype: float64



**Note:** After sampling the data, you may want to write the data to a file for reloading later.

<span style="background: yellow;">If required, remove the `dataset` variable to avoid any memory-related issue.</span> 

In [95]:
# Write your smart, balanced sample to local file  
# ----------------------------------
sampled_X = sample.drop(columns=['went_on_backorder'])
sampled_y = sample['went_on_backorder']
joblib.dump([sampled_X, sampled_y], 'sample-data.pkl')

['sample-data.pkl']